In [1]:
import pandas as pd

In [72]:
train_df = pd.read_csv('../dados/train.csv')
test_df = pd.read_csv('../dados/test.csv')

In [73]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [74]:
train_df.info()
train_df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [75]:
def renomear_colunas(df):
    renomes = {
        'PassengerId': 'Id_Passageiro', # Identificador único do passageiro
        'Survived': 'Sobreviveu',       # 0 = Não, 1 = Sim
        'Pclass': 'ClassePassageiro',   # Classe do passageiro (1 = Alta, 2 = Média, 3 = Baixa)
        'Name': 'Nome',                 # Nome completo
        'Sex': 'Sexo',                  # Gênero
        'Age': 'Idade',                 # Idade em anos
        'SibSp': 'Irmaos_Conjuges',     # Qtd de irmãos ou cônjuges a bordo
        'Parch': 'Pais_Filhos',         # Qtd de pais ou filhos a bordo
        'Ticket': 'NumeroBilhete',      # Número do bilhete
        'Fare': 'ValorBilhete',         # Valor pago na passagem
        'Cabin': 'Cabine',              # Número da cabine
        'Embarked': 'Porto_Embarque',   # C = Cherbourg, Q = Queenstown, S = Southampton
    }
    
    # Retorna o DataFrame com as colunas renomeadas
    return df.rename(columns=renomes)

train_df = renomear_colunas(train_df)

display(train_df.head())

,Id_Passageiro,Sobreviveu,ClassePassageiro,Nome,Sexo,Idade,Irmaos_Conjuges,Pais_Filhos,NumeroBilhete,ValorBilhete,Cabine,Porto_Embarque
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [76]:
from pandasql import sqldf
pysqldf = lambda q: sqldf(q, globals())

query = """
    SELECT ClassePassageiro, Sexo, Porto_Embarque, count(distinct Id_Passageiro) as total
    FROM train_df
    WHERE Idade is null
    group by 1,2,3
    order by total desc
"""

resultado = pysqldf(query)
display(resultado)

,ClassePassageiro,Sexo,Porto_Embarque,total
0,3,male,S,51
1,3,male,Q,25
2,3,female,Q,23
3,3,male,C,18
4,1,male,S,15
5,3,female,S,12
6,2,male,S,7
7,3,female,C,7
8,1,male,C,6
9,1,female,C,5


In [77]:
pysqldf = lambda q: sqldf(q, globals())
query = """
    SELECT ClassePassageiro, Sexo, Porto_Embarque, max(idade) as idade_max, min(idade) as idade_min, median(idade) as idade_mediana
    FROM train_df
    WHERE Idade is not null
    group by 1,2,3
"""

resultado = pysqldf(query)
display(resultado)

,ClassePassageiro,Sexo,Porto_Embarque,idade_max,idade_min,idade_mediana
0,1,female,NaN,62.0,38.00,50.00
1,1,female,C,60.0,16.00,37.00
2,1,female,Q,33.0,33.00,33.00
3,1,female,S,63.0,2.00,33.00
4,1,male,C,71.0,17.00,36.50
5,1,male,Q,44.0,44.00,44.00
6,1,male,S,80.0,0.92,42.00
7,2,female,C,28.0,3.00,22.00
8,2,female,Q,30.0,30.00,30.00
9,2,female,S,57.0,2.00,29.00


In [78]:
# Calcular medianas
medianas_idade_treino = train_df.groupby(['ClassePassageiro', 'Sexo', 'Porto_Embarque'])['Idade'].median()

def preencher_idade(df, medianas_referencia):

    def aplicar_mediana(linha):
        if pd.isnull(linha['Idade']):
            try:
                return medianas_referencia[linha['ClassePassageiro'], linha['Sexo'], linha['Porto_Embarque']]
            except KeyError:
                return medianas_referencia.median() # Plano B de segurança
        return linha['Idade']

    df['Idade'] = df.apply(aplicar_mediana, axis=1)
    return df

# Aplicar no treino
train_df = preencher_idade(train_df, medianas_idade_treino)

In [79]:
train_df.head()

,Id_Passageiro,Sobreviveu,ClassePassageiro,Nome,Sexo,Idade,Irmaos_Conjuges,Pais_Filhos,NumeroBilhete,ValorBilhete,Cabine,Porto_Embarque
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [80]:
pysqldf = lambda q: sqldf(q, globals())
query = """
    SELECT ClassePassageiro, Porto_Embarque, SUBSTR(Cabine, 1, 1) AS Deck, count(distinct Id_Passageiro) as total
    FROM train_df
    where cabine is not null
    group by 1,2,3
    order by 4 desc
"""

resultado = pysqldf(query)
display(resultado)

,ClassePassageiro,Porto_Embarque,Deck,total
0,1,S,C,36
1,1,S,B,23
2,1,C,B,22
3,1,C,C,21
4,1,S,E,20
5,1,S,D,18
6,1,C,D,11
7,1,S,A,8
8,2,S,F,8
9,1,C,A,7


In [81]:
def identificar_deck(df):
    df['Deck'] = df['Cabine'].str[0].fillna('U')
    # Remove a coluna 'Cabine', pois a informação já foi extraída
    df = df.drop('Cabine', axis=1)
       
    return df

# Ajuste aqui: chame a função passando o DataFrame inteiro
train_df = identificar_deck(train_df)

In [ ]:
# descobre o porto moda do treino
porto_moda_treino = train_df['Porto_Embarque'].mode()[0]

def preencher_porto(df, valor_preenchimento):
    # Preenche os espaços vazios com o valor fornecido
    df['Porto_Embarque'] = df['Porto_Embarque'].fillna(valor_preenchimento)
    return df

# aplica funcao
train_df = preencher_porto(train_df, porto_moda_treino)

In [89]:
# descobre mediana da Tarifa por Classe
mediana_fare_treino = train_df.groupby('ClassePassageiro')['ValorBilhete'].median()

def preencher_tarifa(df, mediana_referencia):
    def aplicar_fare(linha):
        if pd.isnull(linha['ValorBilhete']):
            return mediana_referencia[linha['ClassePassageiro']]
        return linha['ValorBilhete']

    df['ValorBilhete'] = df.apply(aplicar_fare, axis=1)
    return df

In [90]:
def transformacoes(train_df, df):
    df = renomear_colunas(df)
    
    # Preencher Idade
    medianas_idade_treino = train_df.groupby(['ClassePassageiro', 'Sexo', 'Porto_Embarque'])['Idade'].median()
    df = preencher_idade(df, medianas_idade_treino)
    
    # Identificar Deck
    df = identificar_deck(df)
    
    # Preencher Porto de Embarque
    
    porto_moda_treino = train_df['Porto_Embarque'].mode()[0]
    df = preencher_porto(df, porto_moda_treino)

    mediana_fare_treino = train_df.groupby('ClassePassageiro')['ValorBilhete'].median()
    df = preencher_tarifa(df, mediana_fare_treino)
    
    return df

In [91]:
test_df = transformacoes(train_df, test_df)

In [93]:
test_df.describe()
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Id_Passageiro     418 non-null    int64  
 1   ClassePassageiro  418 non-null    int64  
 2   Nome              418 non-null    str    
 3   Sexo              418 non-null    str    
 4   Idade             418 non-null    float64
 5   Irmaos_Conjuges   418 non-null    int64  
 6   Pais_Filhos       418 non-null    int64  
 7   NumeroBilhete     418 non-null    str    
 8   ValorBilhete      418 non-null    float64
 9   Porto_Embarque    418 non-null    str    
 10  Deck              418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB
